In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(data_folder: str, **storage_options) -> pd.DataFrame:
    """
    Load the orders table from a CSV-like .tbl file, parsing dates and minimizing memory usage.
    """
    data_path = f"{data_folder}/orders.tbl"
    # Specify dtypes to avoid expensive type inference and reduce memory
    dtype = {
        "O_ORDERKEY": "int64",
        "O_CUSTKEY": "int64",
        "O_ORDERSTATUS": "category",
        "O_TOTALPRICE": "float64",
        "O_ORDERPRIORITY": "category",
        "O_CLERK": "category",
        "O_SHIPPRIORITY": "int64",
        "O_COMMENT": "string"
    }
    # Read in one pass: parse dates in C parser, infer format for speed, memory map if possible
    return pd.read_csv(
        data_path,
        sep="|",
        dtype=dtype,
        parse_dates=["O_ORDERDATE"],
        infer_datetime_format=True,
        memory_map=True,
        storage_options=storage_options
    )

orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")  
    return df
partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Compute ghost part keys
ghost_keys = part.loc[part.P_NAME.str.contains("ghost"), "P_PARTKEY"].to_numpy()

# Build mapping from supplier key to nation name
nation_map = nation.set_index("N_NATIONKEY")["N_NAME"]
supplier_nation = supplier.set_index("S_SUPPKEY")["S_NATIONKEY"].map(nation_map)

# Filter partsupp for ghost parts and attach nation
filtered_ps = partsupp.loc[partsupp.PS_PARTKEY.isin(ghost_keys), ["PS_PARTKEY", "PS_SUPPKEY", "PS_SUPPLYCOST"]]
filtered_ps["N_NAME"] = filtered_ps.PS_SUPPKEY.map(supplier_nation)
filtered_ps = filtered_ps.dropna(subset=["N_NAME"])

# Restrict lineitem columns and rows to ghost parts, then merge with filtered_ps
li_cols = ["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT", "L_QUANTITY"]
li_ghost = lineitem.loc[lineitem.L_PARTKEY.isin(ghost_keys), li_cols]
jn = li_ghost.merge(filtered_ps, left_on=["L_PARTKEY", "L_SUPPKEY"], right_on=["PS_PARTKEY", "PS_SUPPKEY"])

# Merge with orders for order date
jn = jn.merge(orders[["O_ORDERKEY", "O_ORDERDATE"]], left_on="L_ORDERKEY", right_on="O_ORDERKEY")

# Compute profit (TMP) and extract order year
jn["TMP"] = jn.L_EXTENDEDPRICE * (1 - jn.L_DISCOUNT) - jn.PS_SUPPLYCOST * jn.L_QUANTITY
jn["O_YEAR"] = jn.O_ORDERDATE.dt.year

# Group by nation and year, then sort
total = (
    jn.groupby(["N_NAME", "O_YEAR"], as_index=False, sort=False)["TMP"].sum()
      .sort_values(["N_NAME", "O_YEAR"], ascending=[True, False])
)